---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 3.1: Adding the First Tool

## 💬 Product Check-in:

> **Sam (Product):** "Small thing, but users keep asking what the current version of MLflow is and the agent either says it doesn't know or confidently states an old version number. It's hurting the credibility of the agent."
>
> **Sam:** "Also, if we add a tool - how will we know it's actually calling the tool?"

**Objective:** Translate next round of feedback into evaluations and scorers that can be used to refine the agent. Refine aspects of the agent to meet updated evaluation criteria.

In this notebook:
1. Build a `get_mlflow_version` tool
2. Add it to the agent
3. Add `ToolCallCorrectness` and `ToolCallEfficiency` to the eval pipeline
4. Write eval cases with `expected_tool_calls` ground truth

---
## 0 — Connect to MLflow

In [ ]:
from functools import cached_property
from pathlib import Path
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

ENV_FILE = Path.cwd().parent.parent / ".env"  # adjust .parent count to match your nesting

class AgentConfig(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=ENV_FILE,
        env_file_encoding="utf-8",
        extra="ignore",  # don't choke on unrelated vars in .env
    )

    # Secrets / endpoints (from .env)
    gemini_api_key: SecretStr
    gemini_openai_base_url: str

    # MLflow (from .env, with fallbacks)
    mlflow_tracking_uri: str
    experiment_name: str = Field(default="mlflow-agent-1", alias="EXPERIMENT_1_NAME")

    # Model knobs (hardcoded defaults, overridable via env if you want)
    model: str = "gemini-2.5-flash-lite"
    temperature: float = 0.2
    max_tokens: int = 512
    ## Adding judge model to config
    judge_model: str = "gemini:/gemini-3.1-flash-lite-preview"

    # Prompt
    prompt_name: str = "mlflow-agent-system"
    system_prompt: str = "You are an MLflow assistant."

    ##DATASET
    eval_dataset_name: str = "mlflow-agent-eval"

    # Aligned Judge
    align_judge: str = "response_quality"

    @cached_property
    def prompt_alias(self) -> str:
        return f"prompts:/{self.prompt_name}@prod"

cfg = AgentConfig()

In [ ]:
from openai import OpenAI
import mlflow


mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
mlflow.set_experiment(cfg.experiment_name)
mlflow.langchain.autolog()

prompt_version = mlflow.genai.load_prompt(cfg.prompt_alias)
SYSTEM_PROMPT = prompt_version.template

# Convert completions call to langchain agent

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

mlflow.langchain.autolog()

llm = ChatGoogleGenerativeAI(
    model=cfg.model,
    temperature=cfg.temperature,
    max_tokens=cfg.max_tokens,
    base_url=cfg.gemini_openai_base_url
)

tools=[]

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM_PROMPT)

def predict_fn(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result['messages'][-1].content

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is MLflow?" }]})

In [ ]:
version_eval_dataset = [
    # ── Direct version lookup: agent MUST use the PyPI tool ───────────────────
    # No hardcoded version string — we test the *shape* of the answer with a
    # custom scorer (see below) and use expected_facts only for stable truths.
    {
        "inputs": {"question": "What is the current published version of MLflow on PyPI?"},
        "expectations": {
            "expected_facts": [
                "MLflow's current version is on the 3.x release line",
                "The response cites a specific version number in the form 3.X.Y",
            ],
        },
    },

    # ── Stable factual question that benefits from tool grounding ─────────────
    # MLflow 3.0 shipped in June 2024 — this is now a stable historical fact,
    # so expected_facts can pin it down without going stale.
    {
        "inputs": {"question": "Has MLflow 3.0 been released, and if so, when?"},
        "expectations": {
            "expected_facts": [
                "MLflow 3.0 has been released",
                "MLflow 3.0 was released in 2024",
            ],
        },
    },

    # ── Recommendation question: tests that agent recommends 3.x for GenAI ────
    # The fact set is narrow and version-stable: GenAI features require 3.x.
    {
        "inputs": {
            "question": "I want to use mlflow.genai.evaluate and the new tracing features. What major version of MLflow do I need?"
        },
        "expectations": {
            "expected_facts": [
                "MLflow 3 or later is required for the GenAI features",
                "MLflow 2.x does not support the mlflow.genai module",
            ],
        },
    },

    # ── Release notes question: tests the GitHub release notes tool ───────────
    # We don't pin specific feature names (those change every release).
    # Instead we assert that the response references the *latest* release line
    # and cites concrete features rather than generic platitudes.
    {
        "inputs": {"question": "What were the headline features in the most recent MLflow release?"},
        "expectations": {
            "expected_facts": [
                "The response references a specific MLflow 3.x version number",
                "The response names at least one concrete feature from that release",
            ],
        },
    },

    # ── Negative test: question about an old/deprecated API ───────────────────
    # Tests that the agent doesn't blindly recommend pre-3.0 patterns.
    {
        "inputs": {
            "question": "Should I use mlflow.evaluate() for evaluating my GenAI agent in MLflow 3?"
        },
        "expectations": {
            "expected_facts": [
                "mlflow.genai.evaluate is the correct API for GenAI evaluation in MLflow 3",
                "mlflow.evaluate is the older general-purpose evaluation API",
            ],
        },
    },
]

In [ ]:
result['messages'][-1].content

In [ ]:
from mlflow.genai.scorers import Correctness, ExpectationsGuidelines

results_v1 = mlflow.genai.evaluate(
    data=version_eval_dataset,
    predict_fn=predict_fn,
    scorers=[Correctness(model=cfg.judge_model), ExpectationsGuidelines(model=cfg.judge_model)],
)

results_v1

---
## 2 — Building the Version Tool

We'll build a tool that retrieves the latest version from PyPI.

The tool is deterministic and always correct — no hallucination possible. The LLM's job is just to figure out *when* to call it and *what arguments* to pass.

In [ ]:
import requests



def get_mlflow_version_pypi() -> str:
    """
    Fetches the current stable release of MLflow directly from PyPI.
    No API key required.
    """
    try:
        response = requests.get("https://pypi.org/pypi/mlflow/json", timeout=5)
        response.raise_for_status()
        return response.json()["info"]["version"]
    except Exception as e:
        return f"Could not fetch version from PyPI: {str(e)}"

In [ ]:
get_mlflow_version_pypi()

In [ ]:
def get_mlflow_version_pypi() -> str:
    """
    Fetches the current stable release of MLflow directly from PyPI.
    No API key required.
    """
    try:
        response = requests.get("https://pypi.org/pypi/mlflow/json", timeout=5)
        response.raise_for_status()
        return response.json()["info"]["version"]
    except Exception as e:
        return f"Could not fetch version from PyPI: {str(e)}"

---
## 3 — Adding the Tool to the Agent

With the Agents SDK, adding a tool is one line — pass it in the `tools` list. The SDK reads the `@function_tool` docstring and generates the JSON schema for the LLM automatically.

We also update the system prompt to tell the agent about the tool. This matters — the LLM needs to know *when* to use it.

In [ ]:
agent_v2 = create_agent(model=llm, tools=[get_mlflow_version_pypi], system_prompt=SYSTEM_PROMPT)

def predict_fn_v2(question: str) -> str:
    result = agent_v2.invoke({"messages": [{"role": "user", "content": question}]})
    return result['messages'][-1].content

In [ ]:
from mlflow.genai.scorers import Correctness

results_v1 = mlflow.genai.evaluate(
    data=version_eval_dataset,
    predict_fn=predict_fn_v2,
    scorers=[Correctness(model=cfg.judge_model)],
)

results_v1

In [ ]:
import requests
from datetime import datetime

def get_package_info(package: str, max_versions: int = 5) -> dict:
    """Fetch version history + release notes for a PyPI package."""
    # 1. PyPI metadata
    pypi = requests.get(f"https://pypi.org/pypi/{package}/json", timeout=10).json()

    info = pypi["info"]
    releases = pypi["releases"]

    # 2. Build a timestamped version list (skip yanked/empty releases)
    versions = []
    for version, files in releases.items():
        if not files:
            continue
        upload_time = files[0]["upload_time"]
        versions.append((version, upload_time))

    versions.sort(key=lambda v: v[1], reverse=True)
    recent = versions[:max_versions]

    # 3. Try to find a GitHub repo from project_urls
    urls = info.get("project_urls") or {}
    github_repo = _extract_github_repo(urls)

    # 4. Fetch release notes from GitHub if we found a repo
    notes = {}
    if github_repo:
        gh_url = f"https://api.github.com/repos/{github_repo}/releases"
        gh_releases = requests.get(gh_url, timeout=10).json()
        for r in gh_releases:
            # GitHub tags often prefix with 'v' — normalize both sides
            tag = r["tag_name"].lstrip("v")
            notes[tag] = {
                "name": r["name"],
                "body": r["body"],
                "published_at": r["published_at"],
                "url": r["html_url"],
            }

    # 5. Stitch it together
    return {
        "package": package,
        "latest": info["version"],
        "summary": info["summary"],
        "homepage": info.get("home_page"),
        "project_urls": urls,
        "recent_versions": [
            {
                "version": v,
                "released": ts,
                "notes": notes.get(v),
            }
            for v, ts in recent
        ],
    }


def _extract_github_repo(project_urls: dict) -> str | None:
    """Find 'owner/repo' from any github.com URL in project_urls."""
    for url in project_urls.values():
        if url and "github.com/" in url:
            parts = url.split("github.com/")[1].rstrip("/").split("/")
            if len(parts) >= 2:
                return f"{parts[0]}/{parts[1]}"
    return None

In [ ]:
info = get_package_info("mlflow", max_versions=30)
print(f"Latest: {info['latest']}")
for v in info["recent_versions"]:
    print(f"\n── {v['version']} ({v['released']}) ──")
    if v["notes"]:
        print(v["notes"]["body"][:500])
    else:
        print("(no GitHub release notes found)")

In [ ]:
def get_release_notes_from_github() -> str:
    """Get the latest release notes for MLflow from Github"""
    info = get_package_info("mlflow", max_versions=30)
    return info

In [ ]:
agent_v3 = create_agent(model=llm, tools=[get_mlflow_version_pypi, get_release_notes_from_github], system_prompt=SYSTEM_PROMPT)

def predict_fn_v3(question: str) -> str:
    result = agent_v3.invoke({"messages": [{"role": "user", "content": question}]})
    return result['messages'][-1].content

In [ ]:
from mlflow.genai.scorers import Correctness

results_v3 = mlflow.genai.evaluate(
    data=version_eval_dataset,
    predict_fn=predict_fn_v3,
    scorers=[Correctness(model=cfg.judge_model)],
)
#Refine evaluation set here
results_v3

### Making tools visible to MLflow

The key decorator is `@mlflow.trace(span_type=SpanType.TOOL)`. This tells MLflow to record each tool invocation as a `TOOL` span in the trace — capturing inputs, outputs, and timing. Without this, the tool calls would be invisible in the trace.

```python
@mlflow.trace(span_type=SpanType.TOOL)
def my_tool(arg: str) -> dict:
    ...
```

In [ ]:
#add tracing to tool
from mlflow.entities import SpanType


@mlflow.trace(span_type=SpanType.TOOL)
def get_release_notes_from_github() -> str:
    """Get the latest release notes for MLflow from Github"""
    info = get_package_info("mlflow", max_versions=30)
    return info

In [ ]:
agent_v3 = create_agent(model=llm, tools=[get_mlflow_version_pypi, get_release_notes_from_github], system_prompt=SYSTEM_PROMPT)

def predict_fn_v3(question: str) -> str:
    result = agent_v3.invoke({"messages": [{"role": "user", "content": question}]})
    return result['messages'][-1].content

In [ ]:
agent_v3.invoke({"messages": [{"role": "user", "content": "What are some of the new features in MLflow?" }]})

# Span Type Table

Span Types

MLflow defines standard span types:

`CHAIN`: A sequence of operations \
`LLM`: Language model call \
`RETRIEVER`: Document retrieval \
`EMBEDDING`: Text embedding \
`TOOL`: Tool/function execution \
`AGENT`: Agent reasoning \
`PARSER`: Output parsing or generic intermediate parsing of an outcome

Hierarchy Example

```
TRACE (root)
│
└─ SPAN: Agent Executor (AGENT)
   │
   ├─ SPAN: Planning Step (LLM)
   │  └─ attributes: {model: gpt-5, tokens: 50}
   │
   ├─ SPAN: Tool Execution (TOOL)
   │  └─ attributes: {tool: search, query: "..."}
   │
   └─ SPAN: Final Response (LLM)
      └─ attributes: {model: gpt-5, tokens: 150}
```

---
## 4 — Testing the Tool-Calling Agent

Let's test the same question that tripped up the no-tool agent, plus a few more.

### Check the traces

Open the MLflow UI. For the version questions, you should see:

### Add examples to full dataset

---
## 7 — Adding Tool Call Scorers to the Pipeline

We add two new scorers to our existing pipeline:

| Scorer | What it checks | How it works |
|---|---|---|
| `ToolCallCorrectness()` | Did the agent call the right tool with the right arguments? | Compares actual tool calls in the trace against `expected_tool_calls` |
| `ToolCallEfficiency()` | Did the agent make redundant calls? | Checks for duplicate or unnecessary tool invocations |

These scorers read the **trace**, not just the text output. They look for spans with `span_type=TOOL` — which the Agents SDK creates automatically via autologging.

In [ ]:
from mlflow.genai.datasets import get_dataset
retrieved_dataset = get_dataset(name=cfg.eval_dataset_name)

In [ ]:
retrieved_dataset.merge_records(version_eval_dataset)

In [ ]:
from mlflow.genai.scorers import (
    Correctness,
    RelevanceToQuery,
    ToolCallCorrectness,
    ToolCallEfficiency,
)

results = mlflow.genai.evaluate(
    data=retrieved_dataset,
    predict_fn=predict_fn_v3,
    scorers=[
        # ── Tool quality ──────────────────────────────────────────────
        ToolCallCorrectness(model=cfg.judge_model),   # fuzzy match against expected_tool_calls
        ToolCallEfficiency(model=cfg.judge_model),    # no ground truth needed
        # ── Output quality (from Stage 1) ─────────────────────────────
        Correctness(model=cfg.judge_model),
        RelevanceToQuery(model=cfg.judge_model),
    ],
)

print("=== Evaluation with tool call scorers ===")
print(results.metrics)

In [ ]:
results.tables["eval_results_table"]

In [ ]:
prompt = """You are a helpful MLflow assistant.
Be practical and conversational — like a knowledgeable colleague, not a changelog.
Always include concrete code examples with real function signatures and parameter names.
When referencing an API, mention which MLflow version it applies to (e.g. 'new in MLflow 3.0').

Tool usage:
- Use the provided tools when the question concerns MLflow version-specific details like
  recent releases, or version-specific behavior — especially anything
  that may have changed in MLflow 3.x.
- Answer directly from your own knowledge for general programming, Python, machine
  learning fundamentals, or conceptual questions that don't depend on
  MLflow-version-specific details.
- If a tool returns nothing useful, fall back to your own knowledge and clearly
  flag any uncertainty about MLflow 3.x specifics.

Additional guidelines:
- When answering a question, proactively mention related concepts the user should know about
  (e.g. if they ask about logging metrics, mention active runs and experiments)
- Always code examples that illustrate both the direct answer and the related concepts
- Keep responses focused and under 2000 characters
- For safety-sensitive topics (e.g. credentials, API keys), include appropriate caveats"""

In [ ]:
# Register the prompt

In [ ]:
agent_v4 = create_agent(model=llm, tools=[get_mlflow_version_pypi, get_release_notes_from_github], system_prompt=prompt)

def predict_fn_v4(question: str) -> str:
    result = agent_v4.invoke({"messages": [{"role": "user", "content": question}]})
    return result['messages'][-1].content

In [ ]:
from langchain_core.messages import AIMessage

@mlflow.trace(name="qa_agent", span_type="AGENT")
def predict_fn_v4(question: str) -> str:
    result = agent_v4.invoke({
        "messages": [{"role": "user", "content": question}]
    })

    # Walk backward to find the last AIMessage with actual text content.
    # This skips tool messages and AIMessages that are pure tool-call requests.
    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage):
            content = msg.content
            # Handle list-style content blocks (some providers)
            if isinstance(content, list):
                text_parts = [
                    block.get("text", "") for block in content
                    if isinstance(block, dict) and block.get("type") == "text"
                ]
                content = "".join(text_parts)
            if content and content.strip():
                return content

    # Fallback: agent ended without producing text (rare — usually a recursion
    # limit hit). Return something the judge can reason about rather than "".
    return "[Agent did not produce a final text response.]"

In [ ]:
predict_fn_v4("How do I search runs in MLflow")

In [ ]:
prompt_v3 = mlflow.genai.register_prompt(
    name=cfg.prompt_name,
    template=(prompt),
    commit_message="Added related-concept suggestions and safety caveats per Sam's sprint feedback",
    tags={
        "author": "ml-team",
        "agent": "mlflow-assistant",
    },
)

In [ ]:
mlflow.genai.set_prompt_alias(name=cfg.prompt_alias, version=6, alias="prod")

In [ ]:
from mlflow.genai.scorers import (
    Correctness,
    RelevanceToQuery,
    ToolCallCorrectness,
    ToolCallEfficiency,
)

results = mlflow.genai.evaluate(
    data=retrieved_dataset,
    predict_fn=predict_fn_v4,
    scorers=[
        ToolCallCorrectness(model=cfg.judge_model), 
        ToolCallEfficiency(model=cfg.judge_model), 
        Correctness(model=cfg.judge_model),
        RelevanceToQuery(model=cfg.judge_model),
        ExpectationsGuidelines(model=cfg.judge_model)
    ],
)

print("=== Evaluation with tool call scorers ===")
print(results.metrics)